In [ ]:
!pip install -q tifffile
!pip install --upgrade pip
!pip install -q --upgrade tifffile
!pip install -q imagecodecs

In [ ]:
import tifffile
print("tifffile works! version:", tifffile.__version__)


In [ ]:
from pathlib import Path
import math
import tifffile

# 👇 change this to your DEM folder path
DEM_DIR = r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\CROPPED_DEM_10"

def get_px_size(tif: tifffile.TiffFile):
    """Read pixel size (ModelPixelScaleTag) from GeoTIFF tags."""
    tag = tif.pages[0].tags.get(33550)
    if tag:
        sx, sy = tag.value[:2]
        return abs(float(sx)), abs(float(sy))
    return None, None

dem_dir = Path(DEM_DIR)
files = sorted(dem_dir.glob("*.tif"))
print(f"Found {len(files)} DEM tiles\n")

for f in files[:50]:  # check first 10 tiles
    with tifffile.TiffFile(f) as tf:
        page = tf.pages[0]
        w, h = page.imagewidth, page.imagelength
        px_x, px_y = get_px_size(tf)
    print(f"📁 {f.name}")
    print(f"   Pixels: {w} × {h}")
    if px_x and px_y:
        print(f"   Pixel size: {px_x:.3f} × {px_y:.3f} m")
        print(f"   Ground size: {w*px_x:.1f} m × {h*px_y:.1f} m")
        ok = math.isclose(w*px_x, 1000.0, abs_tol=1.0) and math.isclose(h*px_y, 1000.0, abs_tol=1.0)
        print(f"   ~1 km tile? {'YES' if ok else 'NO'}")
    else:
        print("   ⚠️ Pixel size not found in GeoTIFF tags")
    print()


In [ ]:
from pathlib import Path
import numpy as np
import tifffile
from shutil import copy2

# --- paths (same as before) ---
DEM_DIR   = Path(r"D:\Sunita_Thesis\Datasets\ProjectedDEM_10\CROPPED_DEM_10")
MOSAICS = {
    "R":    r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B04.tif",
    "G":    r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B03.tif",
    "B":    r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B02.tif",
    "NIR":  r"D:\Sunita_Thesis\Datasets\switzerland_2016\switzerland_2016\B08.tif",
    "NOISE":r"D:\Sunita_Thesis\Datasets\ProjectedNoise\StrassenLaerm_Tag_LV95_sigma1_32632.tif",
}
OUT_ROOT  = Path(r"D:\Sunita_Thesis\Datasets\Data_Patches_1km_DEMCropped")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

LIMIT = None  # test first; set to None for all

# ---------- helpers ----------
def _read_scale_tie(tf: tifffile.TiffFile):
    tags = tf.pages[0].tags
    scale = tags.get(33550)   # ModelPixelScaleTag
    tie   = tags.get(33922)   # ModelTiepointTag
    if not scale or not tie:
        raise RuntimeError("Missing GeoTIFF ModelPixelScale (33550) or Tiepoint (33922).")
    sx, sy = float(scale.value[0]), float(scale.value[1])
    i0, j0, _, X0, Y0, _ = [float(v) for v in tie.value[:6]]
    return (abs(sx), abs(sy)), (i0, j0), (X0, Y0)

def _origin_px_dims_nodata(tif_path):
    with tifffile.TiffFile(tif_path) as tf:
        (px_x, px_y), (i0, j0), (X0, Y0) = _read_scale_tie(tf)
        w = tf.pages[0].imagewidth
        h = tf.pages[0].imagelength
        # read NoData if present (GDAL_NODATA = 42113)
        nd_tag = tf.pages[0].tags.get(42113)
        nd = None
        if nd_tag:
            val = nd_tag.value
            nd = float(val) if isinstance(val, (bytes, str)) else float(val)
    ox = X0 - i0 * px_x
    oy = Y0 + j0 * px_y
    return (ox, oy), (px_x, px_y), (w, h), nd

def _dem_ul_dims_nodata(dem_path):
    with tifffile.TiffFile(dem_path) as tf:
        (px_x, px_y), (i0, j0), (X0, Y0) = _read_scale_tie(tf)
        w = tf.pages[0].imagewidth
        h = tf.pages[0].imagelength
        nd_tag = tf.pages[0].tags.get(42113)
        nd = None
        if nd_tag:
            val = nd_tag.value
            nd = float(val) if isinstance(val, (bytes, str)) else float(val)
    ulx = X0 + i0 * px_x
    uly = Y0 - j0 * px_y
    return (ulx, uly), (px_x, px_y), (w, h), nd

def _window_from_world(ul_world, size_m, mosaic_origin, px_m):
    ulx, uly = ul_world
    ox, oy   = mosaic_origin
    px_x, px_y = px_m
    col_off = int(round((ulx - ox) / px_x))
    row_off = int(round((oy  - uly) / px_y))  # north-up
    width  = int(round(size_m[0] / px_x))
    height = int(round(size_m[1] / px_y))
    return row_off, col_off, height, width

def _write_geotiff_uncompressed(out_path, array, px, ul_world, geokeys_src_path, nodata=None):
    px_x, px_y = px
    ulx, uly = ul_world
    # copy CRS keys & build tags before closing src
    with tifffile.TiffFile(geokeys_src_path) as src:
        tags = src.pages[0].tags
        geokey = np.array(tags.get(34735).value, dtype=np.uint16) if tags.get(34735) else None
        geodbl = np.array(tags.get(34736).value, dtype=np.float64) if tags.get(34736) else None
        geoasc = tags.get(34737).value
        if isinstance(geoasc, bytes): geoasc = geoasc.decode('ascii', 'ignore')
    extratags = [
        (33550, 'd', 3, (float(px_x), float(px_y), 0.0), False),                 # ModelPixelScale
        (33922, 'd', 6, (0.0, 0.0, 0.0, float(ulx), float(uly), 0.0), False),    # ModelTiepoint
    ]
    if geokey is not None: extratags.append((34735, 'H', geokey.size, geokey, False))
    if geodbl is not None: extratags.append((34736, 'd', geodbl.size, geodbl, False))
    if geoasc is not None: extratags.append((34737, 's', len(geoasc), geoasc, False))
    # add NoData if provided (GDAL_NODATA = 42113)
    if nodata is not None:
        nd_str = str(nodata)
        extratags.append((42113, 's', len(nd_str), nd_str, False))

    tifffile.imwrite(out_path, array, photometric='minisblack', metadata=None,
                     compression=None, extratags=extratags)

def _read_crop_with_padding(src_path, row_off, col_off, height, width, nodata=None):
    """Read a (height,width) window; pad outside with nodata."""
    with tifffile.TiffFile(src_path) as tf:
        src = tf.pages[0].asarray()  # fallback (old tifffile)
    mh, mw = src.shape[:2]
    # destination patch
    if nodata is None:
        # choose a reasonable fill based on dtype
        if np.issubdtype(src.dtype, np.floating):
            nodata = np.nan
        elif np.issubdtype(src.dtype, np.signedinteger):
            nodata = np.iinfo(src.dtype).min
        else:
            nodata = 0
    patch = np.full((height, width), nodata, dtype=src.dtype)

    # intersect with source
    rs0 = max(0, row_off)
    cs0 = max(0, col_off)
    rs1 = min(mh, row_off + height)
    cs1 = min(mw, col_off + width)
    if rs1 > rs0 and cs1 > cs0:
        dr0 = rs0 - row_off
        dc0 = cs0 - col_off
        patch[dr0:dr0 + (rs1 - rs0), dc0:dc0 + (cs1 - cs0)] = src[rs0:rs1, cs0:cs1]
    return patch, nodata
# -------------------------------

# pre-read mosaic metadata
mosaic_meta = {}
for name, p in MOSAICS.items():
    origin, px, dims, nd = _origin_px_dims_nodata(p)
    mosaic_meta[name] = {"origin": origin, "px": px, "dims": dims, "path": p, "nodata": nd}

dem_files = sorted(DEM_DIR.glob("*.tif"))
print(f"Found {len(dem_files)} DEM tiles")

count = 0
for dem_path in (dem_files if LIMIT is None else dem_files[:LIMIT]):
    tile_id = dem_path.stem
    tile_folder = OUT_ROOT / tile_id
    tile_folder.mkdir(parents=True, exist_ok=True)

    # DEM: copy as-is (keeps original NoData/compression)
    copy2(dem_path, tile_folder / f"{tile_id}__DEM.tif")

    # DEM window in world coords = full tile (keep the 102x102 overlap)
    ul_world, dem_px, (dem_w, dem_h), dem_nd = _dem_ul_dims_nodata(dem_path)
    dem_size_m = (dem_w * dem_px[0], dem_h * dem_px[1])

    # crop each mosaic using DEM's full extent; pad outside with nodata
    for name, meta in mosaic_meta.items():
        row_off, col_off, height, width = _window_from_world(
            ul_world=ul_world,
            size_m=dem_size_m,
            mosaic_origin=meta["origin"],
            px_m=meta["px"]
        )
        patch, nd = _read_crop_with_padding(meta["path"], row_off, col_off, height, width, nodata=meta["nodata"])
        _write_geotiff_uncompressed(
            tile_folder / f"{tile_id}__{name}.tif",
            patch, meta["px"], ul_world,
            geokeys_src_path=meta["path"], nodata=nd
        )

    count += 1
    print(f"✓ {tile_id}  →  {tile_folder}")

print(f"Done. Wrote {count} tile folder(s).")
